# 05 - Exportar KPIs de Gold a RDS (MySQL)

## Objetivo
Tomamos los resultados agregados que ya calculamos en `04_joins_agregaciones_spark.ipynb` (los que se encuentran en `gold_data/` en S3) y los copiamos a la base de datos RDS, como tablas pequeñas listas para consultar desde Streamlit.

**Importante - Exportación a RDS:** solo exportamos los 7 resultados agregados finales (cientos o miles de filas cada uno, nunca millones). El dataset completo (`clean_data/`, `gold_data/base_temporal/`) se queda en S3; RDS es solo para KPIs y metadata.

**Kernel: Python 3** — aquí no necesitamos Spark porque los archivos de `gold_data/` ya son pequeños; los leemos directo con pandas.

## Prerrequisitos
Instalamos lo que nos hace falta para que funcione correctamente:
```
pip install pymysql sqlalchemy s3fs pyarrow boto3
```

## Manejo de credenciales
En la celda que valida las credenciales **no tenemos ninguna contraseña escrita**: si ya tenemos `MYSQL_HOST` / `MYSQL_USER` / `MYSQL_DB` como variables de entorno (por ejemplo, si nuestro `.env` ya las carga en la sesión), las vamos a usar automáticamente. Si no las encuentra, nos las va a solicitar en el momento y la contraseña siempre se pide con `getpass`, para que no la muestre en pantalla ni la guarde en el archivo `.ipynb`.

In [2]:
%pip install pymysql sqlalchemy s3fs pyarrow boto3

INFO: pip is looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
  Using cached boto3-1.43.90-py3-none-any.whl.metadata (6.6 kB)
INFO: pip is still looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Attempting uninstall: boto3
    Found existing installation: boto3 1.43.96
    Uninstalling boto3-1.43.96:
      Successfully uninstalled boto3-1.43.96
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import getpass
import json
import boto3
import pandas as pd
from sqlalchemy import create_engine, text

BUCKET = "xideralaws-curso-proyecto-alan"

mysql_host = os.getenv("MYSQL_HOST") or input("Endpoint de RDS MySQL: ")
mysql_port = int(os.getenv("MYSQL_PORT", "3306"))
mysql_user = os.getenv("MYSQL_USER") or input("Usuario de MySQL: ")
mysql_db = os.getenv("MYSQL_DB") or input("Base de datos (schema) donde tienes permiso de crear tablas: ")
mysql_password = os.getenv("MYSQL_PASSWORD") or getpass.getpass("Contraseña de MySQL: ")

connection_string = f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_db}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print("Conexion a RDS exitosa")
print("Host:", mysql_host)
print("Base de datos:", mysql_db)
print("Usuario:", mysql_user)

Endpoint de RDS MySQL:  database-1.cd8w4cuu6a79.us-west-1.rds.amazonaws.com
Usuario de MySQL:  admin
Base de datos (schema) donde tienes permiso de crear tablas:  proyecto_integrador_alan
Contraseña de MySQL:  ········


Conexion a RDS exitosa
Host: database-1.cd8w4cuu6a79.us-west-1.rds.amazonaws.com
Base de datos: proyecto_integrador_alan
Usuario: admin


### Función auxiliar para leer resultados de `gold_data/`
Todos los resultados (excepto la correlación, que guardamos como JSON) son Parquet pequeño, por lo tanto, pandas los puede leer directamente desde S3 sin necesidad de utilizar Spark.

In [2]:
def leer_gold_parquet(nombre_carpeta):
    ruta = f"s3://{BUCKET}/gold_data/{nombre_carpeta}/"
    df = pd.read_parquet(ruta)
    print(f"{nombre_carpeta}: {len(df)} filas leidas")
    return df

s3_client = boto3.client("s3", region_name="us-west-1")

### Exportar las 6 tablas Parquet a RDS
Usamos `if_exists="replace"` a propósito: estos KPIs se recalculan por completo cada vez que corre el pipeline (no se acumulan), así que reemplazar la tabla entera es la forma correcta de mantenerlos sincronizados con la última corrida. Esto la vuelve idempotente gracias al diseño que estamos trabajando, igual que el resto del pipeline.

In [3]:
tablas_parquet = [
    "trips_by_day",
    "peak_hours",
    "weekday_weekend",
    "trip_duration",
    "fare_and_tip",
    "geographic_demand",
]

for nombre in tablas_parquet:
    df = leer_gold_parquet(nombre)
    nombre_tabla = f"gold_{nombre}"
    df.to_sql(nombre_tabla, engine, if_exists="replace", index=False)
    print(f"  -> exportado a tabla RDS: {nombre_tabla}")

trips_by_day: 3569 filas leidas
  -> exportado a tabla RDS: gold_trips_by_day
peak_hours: 96 filas leidas
  -> exportado a tabla RDS: gold_peak_hours
weekday_weekend: 8 filas leidas
  -> exportado a tabla RDS: gold_weekday_weekend
trip_duration: 4 filas leidas
  -> exportado a tabla RDS: gold_trip_duration
fare_and_tip: 3 filas leidas
  -> exportado a tabla RDS: gold_fare_and_tip
geographic_demand: 1049 filas leidas
  -> exportado a tabla RDS: gold_geographic_demand


### Exportar la matriz de correlación
Esta la guardamos como JSON en el notebook 04, no como Parquet. La convertimos a formato "largo" (una fila por par de tipos) para que sea fácil de consultar con SQL normal.

In [4]:
respuesta = s3_client.get_object(Bucket=BUCKET, Key="gold_data/cross_taxi_correlation/correlation_matrix.json")
matriz = json.loads(respuesta["Body"].read())

filas = []
for tipo_a, correlaciones in matriz.items():
    for tipo_b, valor in correlaciones.items():
        filas.append({"taxi_type_a": tipo_a, "taxi_type_b": tipo_b, "correlacion": valor})

df_correlacion = pd.DataFrame(filas)
df_correlacion.to_sql("gold_cross_taxi_correlation", engine, if_exists="replace", index=False)
print(f"gold_cross_taxi_correlation: {len(df_correlacion)} filas exportadas")

gold_cross_taxi_correlation: 16 filas exportadas


### Verificación
Confirma en RDS (no en S3) que cada tabla tiene filas, así nos aseguramos de que no solo se creó la tabla, sino que sí llegamos a los datos.

In [5]:
nombres_tablas = [f"gold_{n}" for n in tablas_parquet] + ["gold_cross_taxi_correlation"]

with engine.connect() as conn:
    for tabla in nombres_tablas:
        resultado = conn.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
        print(f"{tabla}: {resultado} filas en RDS")

gold_trips_by_day: 3569 filas en RDS
gold_peak_hours: 96 filas en RDS
gold_weekday_weekend: 8 filas en RDS
gold_trip_duration: 4 filas en RDS
gold_fare_and_tip: 3 filas en RDS
gold_geographic_demand: 1049 filas en RDS
gold_cross_taxi_correlation: 16 filas en RDS


### Cerrar la conexión

In [6]:
engine.dispose()
print("Conexion a RDS cerrada")

Conexion a RDS cerrada
